In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = pd.read_csv('AAPL.csv', parse_dates=True, index_col='Date')
data.columns = data.columns.str.lower()
data = data.sort_index()

df = data[['close']].copy()
print(df.head())


                close
Date                 
2016-01-04  26.337500
2016-01-05  25.677500
2016-01-06  25.174999
2016-01-07  24.112499
2016-01-08  24.240000


In [3]:
#Returns
df['Return_5'] = df['close'].pct_change(5)
df['Return_10'] = df['close'].pct_change(10)
df['Return_20'] = df['close'].pct_change(20)
#volatility
df['Vol_5'] = df['close'].pct_change().rolling(window=5).std()
df['Vol_10'] = df['close'].pct_change().rolling(window=10).std()
df['Vol_20'] = df['close'].pct_change().rolling(window=20).std()

df['Tomorrow_Return'] = df['close'].pct_change().shift(-1)
df['Signal'] = np.where(df['Tomorrow_Return'] > 0, 1, 0)
df.dropna(inplace=True)
print(df.head())

                close  Return_5  Return_10  Return_20     Vol_5    Vol_10  \
Date                                                                        
2016-02-02  23.620001 -0.055105  -0.022553  -0.103180  0.037028  0.032131   
2016-02-03  24.087500  0.031364  -0.004546  -0.061922  0.021958  0.032853   
2016-02-04  24.150000  0.026677   0.003115  -0.040715  0.022013  0.032811   
2016-02-05  23.504999 -0.034107  -0.072964  -0.025194  0.018563  0.028018   
2016-02-08  23.752501 -0.014726  -0.044549  -0.020111  0.019952  0.028159   

              Vol_20  Tomorrow_Return  Signal  
Date                                           
2016-02-02  0.026989         0.019792       1  
2016-02-03  0.027105         0.002595       1  
2016-02-04  0.026837        -0.026708       0  
2016-02-05  0.025811         0.010530       1  
2016-02-08  0.025904        -0.000211       0  


In [4]:
features = ['Return_5', 'Return_10', 'Return_20', 'Vol_5', 'Vol_10', 'Vol_20']
X = df[features]
y = df['Signal']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0, shuffle=False)
print(X_train.shape, X_valid.shape)

(2084, 6) (521, 6)


In [5]:
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)
print("Logistic Regression Accuracy:", accuracy_score(y_valid, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_valid, y_pred))

Logistic Regression Accuracy: 0.5566218809980806
Confusion Matrix:
 [[  0 231]
 [  0 290]]


In [9]:
n_estimators = [10, 50, 100, 200, 500, 800, 1000, 1200, 1500, 2000]
for n in n_estimators:
    model_2 = RandomForestClassifier(n_estimators=n, random_state=0)
    model_2.fit(X_train, y_train)
    y_pred = model_2.predict(X_valid)
    print(f"Random Forest (n_estimators={n}) Accuracy:", accuracy_score(y_valid, y_pred))

Random Forest (n_estimators=10) Accuracy: 0.5220729366602687
Random Forest (n_estimators=50) Accuracy: 0.5163147792706334
Random Forest (n_estimators=100) Accuracy: 0.508637236084453
Random Forest (n_estimators=200) Accuracy: 0.5163147792706334
Random Forest (n_estimators=500) Accuracy: 0.527831094049904
Random Forest (n_estimators=800) Accuracy: 0.5220729366602687
Random Forest (n_estimators=1000) Accuracy: 0.5201535508637236
Random Forest (n_estimators=1200) Accuracy: 0.5239923224568138
Random Forest (n_estimators=1500) Accuracy: 0.5239923224568138
Random Forest (n_estimators=2000) Accuracy: 0.5239923224568138


In [10]:
model_final = RandomForestClassifier(n_estimators=500, random_state=0)
model_final.fit(X_train, y_train)
y_pred_final = model_final.predict(X_valid)
print("Final Random Forest Accuracy:", accuracy_score(y_valid, y_pred_final))
print("Classification Report:\n", classification_report(y_valid, y_pred_final))

Final Random Forest Accuracy: 0.527831094049904
Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.39      0.43       231
           1       0.57      0.63      0.60       290

    accuracy                           0.53       521
   macro avg       0.51      0.51      0.51       521
weighted avg       0.52      0.53      0.52       521

